In [5]:
import os
import re
import sys
import networkx as nx
import pickle
import pandas as pd
import ast
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline_from_config
import torch
import numpy as np
import json
from pathlib import Path

try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))

from PyKeen.hpo import load_triple_factory

In [6]:
def retrain_with_best_params(triples_factory, output_dir):
    print("\n--- Retraining Best Model Configuration ---")

    ratios = [0.8, 0.1, 0.1]
    train, test, val = triples_factory.split(ratios, random_state=42)
    
    # load best pipeline configs
    pipeline_config = json.loads(
                        Path(f"{output_dir}/best_pipeline/pipeline_config.json").read_text()
                        )
    pipeline_config['pipeline']['training']=train
    pipeline_config['pipeline']['testing']=test
    pipeline_config['pipeline']['validation']=val
    
    best_pipeline_result = pipeline_from_config(pipeline_config)

    # Extract the actual trained PyTorch model object
    best_model = best_pipeline_result.model
    # Save results using the model name
    best_pipeline_result.save_to_directory(output_dir)
    
    print(f"Best Pipeline Results saved to {output_dir}") 
    
    return best_model    

def get_node_embeddings(best_model, triples_factory):
    # Ensure the model is in evaluation mode
    best_model.eval()
    
    with torch.no_grad():
        
        if hasattr(best_model, 'entity_representations') and len(best_model.entity_representations) > 0:
            # Call the representation module to get the actual tensor
            embeddings_tensor = best_model.entity_representations[0]()
        else:
            raise AttributeError("Could not automatically locate entity embeddings on this model.")
        
        # Convert to NumPy array for easy use with Scikit-Learn
        embeddings_ndarray = embeddings_tensor.detach().numpy()
    
    print(f"Extracted embeddings shape: {embeddings_ndarray.shape}")
    
    # Create a mapping from entity label/ID to its embedding row index
    entity_to_id = triples_factory.entity_to_id
    
    return embeddings_ndarray, entity_to_id

Main() script

In [ ]:
# 1. convert graph to TiplesFactory and split data
graph_file = "../datasets/Prime_KGs/G_adni_ADKG_all.pkl"
config_dir = "../PyKeen/results/PrimeKG/adni/all/ADKG/RotatE"
tf = load_triple_factory(str(graph_file))

# 2. retrain model
best_model = retrain_with_best_params(triples_factory=tf, 
                                      output_dir=config_dir)


Load Graph with 5364 nodes and 751002 edges.


No random seed is specified. Setting to 1264827434.
No cuda devices were available. The model runs on CPU
INFO:pykeen.triples.triples_factory:Creating inverse triples.


Number of entities: 5364
Number of relations: 28

--- Retraining Best Model Configuration ---


Training epochs on cpu: 100%|██████████| 10/10 [11:18<00:00, 67.85s/epoch, loss=0.343, prev_loss=0.343]
Evaluating on cpu:   0%|          | 0.00/38.4k [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 38.4k/38.4k [02:26<00:00, 262triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 148.28s seconds
INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=5364, num_relations=28, create_inverse_triples=True, num_triples=307417) to file:///Users/yuxiaoxuan/master_thesis/HybridKG/PyKeen/results/PrimeKG/adni/all/ADKG/RotatE/training_triples
INFO:pykeen.pipeline.api:Saved to directory: /Users/yuxiaoxuan/master_thesis/HybridKG/PyKeen/results/PrimeKG/adni/all/ADKG/RotatE


Best Pipeline Results saved to ../PyKeen/results/PrimeKG/adni/all/ADKG/RotatE


In [20]:
# 3. get embeddings
node_embeddings, entity_to_id = get_node_embeddings(best_model, tf)

# graph_summary_table
design_path = "../AD/data/ADNI/sample_scoring/sample_scoring_all.csv"
design_df = pd.read_csv(design_path, index_col=0)
node_labels_map = design_df['label'].to_dict()
print(node_labels_map)

X = []  # Features (embeddings)
y = []  # Targets (labels)

for entity_name, label in node_labels_map.items():
    if entity_name in entity_to_id:
        # Get the row index of this entity in the embedding matrix
        entity_idx = entity_to_id[entity_name]
        
        # Append the embedding vector and the label
        X.append(node_embeddings[entity_idx])
        y.append(label)

X = np.array(X)
y = np.array(y)

# Check if the data is complex, and concatenate real + imaginary parts if it is
if np.iscomplexobj(X):
    print("Detected complex embeddings. Converting to real-valued features...")
    # Concatenate the real part and imaginary part along the feature axis (axis=1)
    X = np.hstack([X.real, X.imag])

print(f"Prepared {X.shape[0]} samples with {X.shape[1]} features for classification.")

print(f"Prepared {len(X)} samples for classification.")

Extracted embeddings shape: (5364, 64)
{'116_S_1249': 1, '037_S_4410': 0, '006_S_4153': 1, '116_S_1232': 0, '128_S_0205': 0, '036_S_4491': 0, '031_S_2018': 0, '067_S_4072': 0, '037_S_4308': 0, '041_S_4200': 1, '018_S_4313': 0, '067_S_0257': 0, '114_S_4404': 1, '116_S_4167': 1, '116_S_4209': 1, '128_S_1407': 1, '022_S_1097': 1, '014_S_4615': 1, '002_S_1268': 1, '024_S_0985': 1, '073_S_4382': 0, '067_S_4310': 0, '041_S_4427': 0, '129_S_4396': 0, '019_S_4477': 1, '041_S_4271': 1, '003_S_1122': 1, '012_S_4094': 1, '136_S_0186': 0, '098_S_4050': 0, '073_S_4155': 0, '013_S_4580': 0, '100_S_1286': 0, '094_S_2201': 0, '016_S_4097': 0, '023_S_0376': 1, '024_S_4084': 0, '031_S_4496': 0, '014_S_4079': 1, '072_S_4102': 1, '032_S_0677': 0, '127_S_0259': 1, '037_S_0467': 1, '006_S_4192': 1, '137_S_0668': 0, '002_S_2073': 0, '123_S_4526': 1, '003_S_4555': 0, '010_S_4442': 0, '035_S_4082': 0, '037_S_4015': 1, '098_S_4275': 0, '068_S_4174': 0, '068_S_0473': 0, '010_S_4345': 0, '041_S_1010': 1, '014_S_0

In [25]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, 
                            precision_recall_fscore_support, 
                            classification_report,
                            roc_auc_score, 
                            precision_recall_curve, 
                            auc)
from sklearn.preprocessing import StandardScaler

# --- Define Models and Hyperparameter Grids ---
# use relatively small grids here for efficiency
model_configs = {
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=1000, random_state=42),
        "params": {
            "C": [0.1, 1.0, 10.0],
            "penalty": ["l2"]
        }
    },
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [50, 100, 200],
            "max_depth": [None, 10, 20]
        }
    },
    "MLP": {
        "model": MLPClassifier(max_iter=500, random_state=42),
        "params": {
            "hidden_layer_sizes": [(64,), (128, 64)],
            "alpha": [0.0001, 0.001]
        }
    }
}

def calculate_auprc(y_true, y_prob, num_classes):
    """Calculates Macro AUPRC supporting both binary and multiclass data."""
    if num_classes <= 2:
        # Binary case: y_prob is expected to be the probabilities of the positive class
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        return auc(recall, precision)
    else:
        # Multiclass case: Calculate PR-AUC per class and average them (Macro)
        y_true_bin = label_binarize(y_true, classes=np.unique(y_true))
        auprc_list = []
        for i in range(num_classes):
            precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_prob[:, i])
            auprc_list.append(auc(recall, precision))
        return np.mean(auprc_list)

def gridSearchCV(X,y):
    # --- Initialize Cross-Validation ---
    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # Dictionary to store structured metrics per fold for each model
    all_results = {}

    print("\n--- Starting HPO and Cross-Validation Pipeline ---")

    for model_name, config in model_configs.items():
        print(f"\nEvaluating Model: {model_name}")
        all_results[model_name] = []
        
        # Track metrics across folds to print an average later
        fold_accuracies = []
        
        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            
            # Scaling embeddings is highly recommended for Logistic Regression, SVM, and MLP
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            # Inner loop: HPO via Grid Search
            grid_search = GridSearchCV(
                estimator=config["model"],
                param_grid=config["params"],
                cv=3, 
                scoring="accuracy",
                n_jobs=-1
            )
            grid_search.fit(X_train_scaled, y_train)
            
            # Predict on the outer fold test set using the best estimator found
            best_clf = grid_search.best_estimator_
            
            # Predictions
            y_pred = best_clf.predict(X_test_scaled)
            y_prob = best_clf.predict_proba(X_test_scaled) # Get probability scores
            
            # Calculate standard metrics
            acc = accuracy_score(y_test, y_pred)
            precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
            
            # Calculate AUROC & AUPRC based on setup dimensions
            num_classes = max(y)+1
            if num_classes <= 2:
                # For binary, pass probabilities of the positive class only
                auroc = roc_auc_score(y_test, y_prob[:, 1])
                auprc = calculate_auprc(y_test, y_prob[:, 1], num_classes)
            else:
                # For multiclass, pass the full probability matrix using 'ovo' or 'ovr'
                auroc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
                auprc = calculate_auprc(y_test, y_prob, num_classes)
                
            fold_accuracies.append(acc)
            
            # Save comprehensive fold metrics
            fold_metrics = {
                "fold": fold,
                "accuracy": acc,
                "precision_macro": precision,
                "recall_macro": recall,
                "f1_macro": f1,
                "auroc_macro": auroc,
                "auprc_macro": auprc,
                "best_params": grid_search.best_params_
            }
            all_results[model_name].append(fold_metrics)
            
            print(f"  -> Fold {fold}/{n_splits} | Best Params: {grid_search.best_params_} | Accuracy: {acc:.4f}")

        print(f"Mean {model_name} Accuracy: {np.mean(fold_accuracies):.4f}")
    
    return all_results

In [26]:
all_results = gridSearchCV(X,y)


--- Starting HPO and Cross-Validation Pipeline ---

Evaluating Model: LogisticRegression
  -> Fold 1/5 | Best Params: {'C': 0.1, 'penalty': 'l2'} | Accuracy: 0.6044
  -> Fold 2/5 | Best Params: {'C': 0.1, 'penalty': 'l2'} | Accuracy: 0.6154


/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.1

  -> Fold 3/5 | Best Params: {'C': 0.1, 'penalty': 'l2'} | Accuracy: 0.4945
  -> Fold 4/5 | Best Params: {'C': 0.1, 'penalty': 'l2'} | Accuracy: 0.5495
  -> Fold 5/5 | Best Params: {'C': 0.1, 'penalty': 'l2'} | Accuracy: 0.5714
Mean LogisticRegression Accuracy: 0.5670

Evaluating Model: RandomForest
  -> Fold 1/5 | Best Params: {'max_depth': 10, 'n_estimators': 50} | Accuracy: 0.5824
  -> Fold 2/5 | Best Params: {'max_depth': None, 'n_estimators': 50} | Accuracy: 0.5275
  -> Fold 3/5 | Best Params: {'max_depth': 10, 'n_estimators': 200} | Accuracy: 0.4835
  -> Fold 4/5 | Best Params: {'max_depth': None, 'n_estimators': 100} | Accuracy: 0.5165
  -> Fold 5/5 | Best Params: {'max_depth': 10, 'n_estimators': 100} | Accuracy: 0.5275
Mean RandomForest Accuracy: 0.5275

Evaluating Model: MLP
  -> Fold 1/5 | Best Params: {'alpha': 0.001, 'hidden_layer_sizes': (128, 64)} | Accuracy: 0.5385
  -> Fold 2/5 | Best Params: {'alpha': 0.0001, 'hidden_layer_sizes': (128, 64)} | Accuracy: 0.5165
  -> Fo

In [ ]:
# --- Export and View Metrics as a DataFrame ---
print("\n--- Summary of All Folds ---")
all_df = []
for model_name, folds_data in all_results.items():
    df_metrics = pd.DataFrame(folds_data)
    df_metrics['model'] = [model_name]*len(folds_data)
    
    # print(f"\nResults for {model_name}:")
    # print(df_metrics.to_string(index=False))
    all_df.append(df_metrics)
all_df


--- Summary of All Folds ---

Results for LogisticRegression:
 fold  accuracy  precision_macro  recall_macro  f1_macro  auroc_macro  auprc_macro                 best_params              model
    1  0.604396         0.603741      0.603240  0.603198     0.639265     0.618251 {'C': 0.1, 'penalty': 'l2'} LogisticRegression
    2  0.615385         0.617857      0.611702  0.608578     0.585106     0.578666 {'C': 0.1, 'penalty': 'l2'} LogisticRegression
    3  0.494505         0.493956      0.493956  0.493956     0.527563     0.552034 {'C': 0.1, 'penalty': 'l2'} LogisticRegression
    4  0.549451         0.553571      0.550725  0.543943     0.542029     0.599095 {'C': 0.1, 'penalty': 'l2'} LogisticRegression
    5  0.571429         0.573485      0.570290  0.566190     0.596618     0.603350 {'C': 0.1, 'penalty': 'l2'} LogisticRegression

Results for RandomForest:
 fold  accuracy  precision_macro  recall_macro  f1_macro  auroc_macro  auprc_macro                              best_params       

In [ ]:
dfs_to_combine = []

for model_name, folds_data in all_results.items():

    df_temp = pd.DataFrame(folds_data)
    # Ensure the model name column is included if it isn't already
    if 'model' not in df_temp.columns:
        df_temp['model'] = model_name
        
    dfs_to_combine.append(df_temp)

# 3. Concatenate them vertically into one master DataFrame
combined_df = pd.concat(dfs_to_combine, ignore_index=True)

# 4. Clean up the column order for presentation
column_order = [
    "model", "fold", "accuracy", "precision_macro", 
    "recall_macro", "f1_macro", "auroc_macro", "auprc_macro", "best_params"
]
combined_df = combined_df[column_order]
# View the result
combined_df

,model,fold,accuracy,precision_macro,recall_macro,f1_macro,auroc_macro,auprc_macro,best_params
0,LogisticRegression,1,0.604396,0.603741,0.603240,0.603198,0.639265,0.618251,"{'C': 0.1, 'penalty': 'l2'}"
1,LogisticRegression,2,0.615385,0.617857,0.611702,0.608578,0.585106,0.578666,"{'C': 0.1, 'penalty': 'l2'}"
2,LogisticRegression,3,0.494505,0.493956,0.493956,0.493956,0.527563,0.552034,"{'C': 0.1, 'penalty': 'l2'}"
3,LogisticRegression,4,0.549451,0.553571,0.550725,0.543943,0.542029,0.599095,"{'C': 0.1, 'penalty': 'l2'}"
4,LogisticRegression,5,0.571429,0.573485,0.570290,0.566190,0.596618,0.603350,"{'C': 0.1, 'penalty': 'l2'}"
5,RandomForest,1,0.582418,0.582576,0.579062,0.576225,0.593810,0.594626,"{'max_depth': 10, 'n_estimators': 50}"
6,RandomForest,2,0.527473,0.529902,0.529497,0.526558,0.533124,0.565991,"{'max_depth': None, 'n_estimators': 50}"
7,RandomForest,3,0.483516,0.481707,0.481867,0.481261,0.474855,0.482921,"{'max_depth': 10, 'n_estimators': 200}"
8,RandomForest,4,0.516484,0.516683,0.516667,0.516425,0.555314,0.572903,"{'max_depth': None, 'n_estimators': 100}"
9,RandomForest,5,0.527473,0.527864,0.526087,0.519110,0.506280,0.557787,"{'max_depth': 10, 'n_estimators': 100}"
